# R2: Ablation Studies — Shuffled-Rationale & Plain-Continuation

**Purpose:** Test whether rationale SEMANTICS matter or if Stage 2 improvement comes from extra training.

Two ablations (both use seed=42):
1. **Shuffled-rationale:** Stage 2 with randomly reassigned rationales (same format, wrong semantics)
2. **Plain-continuation:** Stage 2 on same 1,221 texts in classification-only format (no rationale/implied)

**Interpreting results:**
- If Shuffled ~ RSQwen and Plain < RSQwen -> CoT format helps, not semantics
- If Shuffled < RSQwen -> Rationale semantics DO matter
- If Plain ~ RSQwen -> Just extra training helps, not even the format

**Output:** `results_ablations.json`

**Estimated time:** 5-7h on A100

## Step 1: Install Dependencies

In [ ]:
!pip uninstall -y protobuf
!pip install -q protobuf==3.20.0
!pip install -q transformers datasets accelerate sentencepiece
!pip install -q scikit-learn pandas numpy tqdm openpyxl
!pip uninstall -y bitsandbytes
!pip install -U bitsandbytes
!pip install -q peft

import os, warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['GRPC_VERBOSITY'] = 'ERROR'
warnings.filterwarnings('ignore')
print('Dependencies installed')

## Step 2: Mount Drive & Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, sys, os, json
from pathlib import Path

DRIVE_BASE = Path('/content/drive/MyDrive/ViTHSD')
WORK_DIR = Path('/content/vithsd')
WORK_DIR.mkdir(exist_ok=True)

for f in ['config.py', 'data_preparation.py', 'evaluation.py', 'models.py']:
    shutil.copy2(DRIVE_BASE / 'src' / f, WORK_DIR / f)

os.chdir(WORK_DIR)
sys.path.insert(0, str(WORK_DIR))

OUTPUT_DIR = DRIVE_BASE / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

import config
config.DATA_DIR = DRIVE_BASE / 'data'
config.OUTPUT_DIR = OUTPUT_DIR
config.BASE_DIR = WORK_DIR

print('Setup complete')

In [ ]:
import torch, random, gc
import numpy as np

gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')

BATCH_SIZE = 8 if gpu_mem >= 35 else (4 if gpu_mem >= 14 else 2)
SEED = 42
print(f'Batch size: {BATCH_SIZE}')

## Step 3: Load Data

In [ ]:
from config import FINAL_LABELS
from data_preparation import load_dataset_A
from evaluation import Evaluator
from models import create_model

train_texts, train_labels = load_dataset_A('train', data_dir=config.DATA_DIR)
test_texts, test_labels = load_dataset_A('test', data_dir=config.DATA_DIR)

with open(DRIVE_BASE / 'data' / 'dataset_rationale.json', 'r', encoding='utf-8') as f:
    rationale_data = json.load(f)

r_train_texts, r_train_implied, r_train_rationale, r_train_labels = [], [], [], []
for r in rationale_data:
    if str(r.get('dataset', '')).lower().strip() != 'train':
        continue
    content = (r.get('content') or '').strip()
    implied = (r.get('implied_statement') or '').strip()
    if not content or not implied:
        continue
    r_train_texts.append(content)
    r_train_implied.append(implied)
    r_train_rationale.append(r.get('rationale', []))
    r_train_labels.append(r.get('labels', []))

alignment_pairs = [{'content': c, 'implied': i} for c, i in zip(r_train_texts, r_train_implied)]
print(f'Train: {len(train_texts)}, Test: {len(test_texts)}, Rationale pairs: {len(alignment_pairs)}')

## Step 4: Helper Functions

In [ ]:
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import time

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def stage2_train(model, training_texts, num_epochs=1, learning_rate=5e-5):
    """Generic Stage 2 training on pre-formatted texts."""
    class TextDataset(Dataset):
        def __init__(self, texts, tokenizer, max_length):
            self.encodings = tokenizer(
                texts, truncation=True, padding=True,
                max_length=max_length, return_tensors='pt'
            )
        def __len__(self):
            return len(self.encodings['input_ids'])
        def __getitem__(self, idx):
            return {
                'input_ids': self.encodings['input_ids'][idx],
                'attention_mask': self.encodings['attention_mask'][idx]
            }

    dataset = TextDataset(training_texts, model.tokenizer, model.max_length)
    dataloader = DataLoader(dataset, batch_size=model.batch_size, shuffle=True)
    optimizer = torch.optim.AdamW(model.model.parameters(), lr=learning_rate)

    model.model.train()
    for epoch in range(num_epochs):
        total_loss = 0
        progress = tqdm(dataloader, desc=f'  Stage2 Ep {epoch+1}/{num_epochs}')
        for batch in progress:
            input_ids = batch['input_ids'].to(model.device)
            attention_mask = batch['attention_mask'].to(model.device)
            optimizer.zero_grad()
            outputs = model.model(
                input_ids=input_ids, attention_mask=attention_mask, labels=input_ids
            )
            loss = outputs.loss
            total_loss += loss.item()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.model.parameters(), 1.0)
            optimizer.step()
            progress.set_postfix({'loss': f'{loss.item():.4f}'})
        print(f'  Epoch {epoch+1}: avg_loss = {total_loss / len(dataloader):.4f}')


def cleanup_model(model):
    if hasattr(model, 'model') and model.model is not None:
        del model.model
    if hasattr(model, 'tokenizer') and model.tokenizer is not None:
        del model.tokenizer
    del model
    gc.collect()
    torch.cuda.empty_cache()


def convert_to_serializable(obj):
    if isinstance(obj, dict):
        return {k: convert_to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [convert_to_serializable(item) for item in obj]
    elif isinstance(obj, (np.integer, np.floating)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

print('Helper functions defined')

## Step 5: Prepare Ablation Data

**Ablation A - Shuffled-Rationale:** Each sample gets another sample's rationale/implied.
Same CoT format, but semantically unrelated rationale.

**Ablation B - Plain-Continuation:** Same 1,221 texts in standard classification format (no CoT).

In [ ]:
set_seed(SEED)
n = len(r_train_texts)

# Create a derangement (no element maps to itself)
shuffled_indices = list(range(n))
random.shuffle(shuffled_indices)
for i in range(n):
    if shuffled_indices[i] == i:
        j = (i + 1) % n
        shuffled_indices[i], shuffled_indices[j] = shuffled_indices[j], shuffled_indices[i]

shuffled_implied = [r_train_implied[shuffled_indices[i]] for i in range(n)]
shuffled_rationale = [r_train_rationale[shuffled_indices[i]] for i in range(n)]

same_count = sum(1 for i in range(n) if shuffled_indices[i] == i)
print(f'Shuffled: {n} pairs, {same_count} self-matches (should be 0)')
print(f'Plain: {n} texts in classification format')

## Step 6: Ablation A — Shuffled-Rationale

In [ ]:
print(f'{"="*70}')
print('ABLATION A: Shuffled-Rationale Stage 2')
print(f'{"="*70}')
t0 = time.time()

set_seed(SEED)
evaluator = Evaluator()

model_shuf = create_model(
    'qwen', dataset_type='A',
    batch_size=BATCH_SIZE, num_epochs=2,
    learning_rate=2e-4, lora_r=8, lora_alpha=16
)

# Stage 1 (identical to RSQwen)
print('  Stage 1...')
set_seed(SEED)
model_shuf.train(train_texts, train_labels)

# Stage 2 with SHUFFLED rationale
print('  Stage 2 (shuffled rationale)...')
set_seed(SEED)

shuffled_texts = []
for i in range(len(r_train_texts)):
    input_text = model_shuf._format_input_cot(r_train_texts[i])
    output_text = model_shuf._format_output_cot(
        r_train_labels[i],       # ORIGINAL labels
        shuffled_rationale[i],    # SHUFFLED rationale
        shuffled_implied[i]       # SHUFFLED implied
    )
    shuffled_texts.append(input_text + output_text)

stage2_train(model_shuf, shuffled_texts, num_epochs=1, learning_rate=5e-5)

print('  Predicting...')
pred_shuf, raw_shuf = model_shuf.predict(test_texts)
res_shuf = evaluator.evaluate(test_labels, pred_shuf, 'Shuffled-Rationale', 'ablation')

elapsed = (time.time() - t0) / 60
print(f'  DONE in {elapsed:.1f} min | F1-Micro={res_shuf["f1_micro"]:.4f}, F1-Macro={res_shuf["f1_macro"]:.4f}')

cleanup_model(model_shuf)

## Step 7: Ablation B — Plain-Continuation

In [ ]:
print(f'{"="*70}')
print('ABLATION B: Plain-Continuation Stage 2')
print(f'{"="*70}')
t0 = time.time()

set_seed(SEED)
evaluator = Evaluator()

model_plain = create_model(
    'qwen', dataset_type='A',
    batch_size=BATCH_SIZE, num_epochs=2,
    learning_rate=2e-4, lora_r=8, lora_alpha=16
)

# Stage 1 (identical to RSQwen)
print('  Stage 1...')
set_seed(SEED)
model_plain.train(train_texts, train_labels)

# Stage 2 with classification-only format (NO rationale/implied)
print('  Stage 2 (plain continuation, no CoT)...')
set_seed(SEED)

plain_texts = []
for i in range(len(r_train_texts)):
    input_text = model_plain._format_input_standard(r_train_texts[i])
    output_text = model_plain._format_output_standard(r_train_labels[i])
    plain_texts.append(input_text + output_text)

stage2_train(model_plain, plain_texts, num_epochs=1, learning_rate=5e-5)

print('  Predicting...')
pred_plain, raw_plain = model_plain.predict(test_texts)
res_plain = evaluator.evaluate(test_labels, pred_plain, 'Plain-Continuation', 'ablation')

elapsed = (time.time() - t0) / 60
print(f'  DONE in {elapsed:.1f} min | F1-Micro={res_plain["f1_micro"]:.4f}, F1-Macro={res_plain["f1_macro"]:.4f}')

cleanup_model(model_plain)

## Step 8: Summary & Save

In [ ]:
print(f'{"="*70}')
print(f'{"ABLATION RESULTS SUMMARY":^70}')
print(f'{"="*70}')
print(f'{"":>22} {"F1-Micro":>12} {"F1-Macro":>12} {"Accuracy":>12}')
print(f'{"-"*60}')
print(f'{"Shuffled-Rationale":>22} {res_shuf["f1_micro"]*100:10.2f}% {res_shuf["f1_macro"]*100:10.2f}% {res_shuf["subset_accuracy"]*100:10.2f}%')
print(f'{"Plain-Continuation":>22} {res_plain["f1_micro"]*100:10.2f}% {res_plain["f1_macro"]*100:10.2f}% {res_plain["subset_accuracy"]*100:10.2f}%')
print(f'\n(Compare with RSQwen and Vanilla from R1 results_multiseed.json)')

ablation_results = {
    'seed': SEED,
    'shuffled_rationale': convert_to_serializable(res_shuf),
    'plain_continuation': convert_to_serializable(res_plain),
    'shuffled_predictions': pred_shuf,
    'plain_predictions': pred_plain
}

with open(OUTPUT_DIR / 'results_ablations.json', 'w') as f:
    json.dump(ablation_results, f, indent=2, ensure_ascii=False)

print(f'\nSaved: {OUTPUT_DIR / "results_ablations.json"}')
print('DONE! Download results_ablations.json from Drive.')